In [ ]:
!pip install dbfread
!pip install POT
!pip install alphashape

In [ ]:
# Import basic libraries
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import math
from matplotlib import colors
from typing import List

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Importing gzip and fits for opening files
import gzip
from io import StringIO
# !pip install dbfread
import dbfread
from dbfread import DBF
import urllib.request, json
import networkx as nx
import seaborn as sns
# !pip install POT
import ot
from scipy.optimize import linear_sum_assignment
# !pip install alphashape
import alphashape
from matplotlib.path import Path
from matplotlib.patches import PathPatch

# Importing OT dependencies
from scipy.spatial.distance import cdist
from ot import emd2
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs

import networkx as nx

#plt.style.use('seaborn')

In [ ]:
from types import TracebackType
# Hashmap for each state (numbers are for the URL)
#Is Mississippi the most segregated state for black people?
#Bama 2
#
states_map = {
    1 : "AL", 2 : "AK", 4 : "AZ", 5 : "AR", 6 : "CA", 8 : "CO", 9 : "CT",
    10 : "DE", 12 : "FL", 13 : "GA", 15: "HI", 16 : "ID", 17 : "IL",
    18 : "IN", 19 : "IA", 20 : "KS", 21 : "KY", 22 : "LA", 23 : "ME",
    24 : "MD", 25: "MA", 26 : "MI", 27 : "MN", 28 : "MS", 29 : "MO",
    30 : "MT", 31: "NE", 32 : "NV", 33 : "NH", 34 : "NJ", 35 : "NM",
    36 : "NY", 37 : "NC", 38 :"ND", 39 : "OH", 40 : "OK", 41 : "OR",
    42 : "PA", 44 : "RI", 45 : "SC", 46 : "SD", 47 : "TN", 48 : "TX",
    49 : "UT", 50 : "VT", 51 : "VA", 53 : "WA", 54 : "WV", 55 : "WI",
    56 : "WY"
}

states_map_2 = {v: k for k, v in states_map.items()}

states_array = [1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56]

# Demographics list from the 2010 census. Everything below this line is
# used to grab information about demographics from the .json file
r_original = {
    1 : "_BLACK",
    2 : "_WHITE",
    3 : "_ASIAN",
    4 : "HISP"
}

r_map = {}
for i in range(1, 4):
    string_arr = ['NH', 'H']
    result = [s + r_original[i] for s in string_arr]
    r_map[i] = result

r_map[4] = ['HISP']

# Initializes URL map in the COUNTY level (less nodes)
c_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/COUNTY/COUNTY_"
    if(i < 10):
        index_str = str(0) + index_str
    c_map[i] = url + index_str + ".json"

tract_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/BLOCK/BLOCK_"
    if(i < 10):
        index_str = str(0) + index_str
    tract_map[i] = url + index_str + ".json"
# Initializes URL map in the TRACT level (more nodes)
url_map = {}
for i in range(1, 57):
    index_str = str(i)
    url = "https://people.csail.mit.edu/ddeford/TRACT/TRACT_"
    if(i < 10):
        index_str = str(0) + index_str
    url_map[i] = url + index_str + ".json"

In [ ]:
# Function to sum the total population of a certain demographic within a census unit
# Input:
#   - data : data["nodes"][ID], where ID is the county or tract ID from the file
#   - r1   : demographic array
#            example: r_map[1] is the demographic array for the black population
#
# Output:
#   - integer value for total population of a certain demographic within the
#     given geographic unit
def sum_total(data, r1):
    result = 0
    for i in range(0, len(r1)):
        result += data[r1[i]]

    return result

In [ ]:
# Function to create an array of cartesian coordinates for each
# census unit, along with a proportion for the desired demographic
#
# Input:
#   - url : a url, which can be taken from url_map
#   - r1 : a demographic array, taken from r_map
#   - url2 : a county url, taken from c_map
#
# Output:
#   - an array structured as follows:
#     [x coordinates, y coordinates, proportion of demographic]
def load_array(url, r1, url2):
    response = urllib.request.urlopen(url)
    data = json.loads(response.read())


    node_data = data["nodes"]


    x_arr_1 = []
    y_arr_1 = []
    prop_arr_1 = []
    total_pop_arr_1 = []
    total_pop = 0

    for i in range(0, len(node_data)):
        x_arr_1.append(node_data[i]['C_X'])
        y_arr_1.append(node_data[i]['C_Y'])
        if(node_data[i]['TOTPOP'] == 0):
            prop_arr_1.append(0)
            total_pop_arr_1.append(0)
        else:
            total_r = sum_total(node_data[i], r1)
            total_pop = node_data[i]['TOTPOP']
            prop_arr_1.append(total_r/total_pop)
            total_pop_arr_1.append(total_pop)

    return [x_arr_1, y_arr_1, prop_arr_1, total_pop_arr_1]

In [ ]:
'''
I want to create a method that will load an array in the same format for a singular county.

INPUT:
url - Block url from which data is taken
county_num - The number of the county
r1 - Demographic array

OUTPUT:
an array structured as follows:
[x coordinates, y coordinates, proportion of demographic, total_pop]
'''


def load_within_county(url, county_num, r1):
    response = urllib.request.urlopen(url)
    data = json.loads(response.read())

    node_data = data["nodes"]
    prop_arr_1 = []
    total_pop_arr_1 = []

    x_arr_1 = []
    y_arr_1 = []

    for i in range(len(node_data)):
        if node_data[i]["COUNTYFP10"] == county_num:
            x_arr_1.append(node_data[i]['C_X'])
            y_arr_1.append(node_data[i]['C_Y'])
            if(node_data[i]['TOTPOP'] == 0):
                prop_arr_1.append(0)
                total_pop_arr_1.append(0)
            else:
                total_r = sum_total(node_data[i], r1)
                total_pop = node_data[i]['TOTPOP']
                prop_arr_1.append(total_r/total_pop)
                total_pop_arr_1.append(total_pop)
    return [x_arr_1, y_arr_1, prop_arr_1, total_pop_arr_1]

In [ ]:
def create_all_county_maps(r1, state_num):
    county_url = c_map[state_num]
    print(county_url)
    response = urllib.request.urlopen(county_url)
    data = json.loads(response.read())

    all_maps = []

    node_data = data["nodes"]

    for i in data["nodes"]:
        all_maps.append(load_within_county(tract_map[state_num], i["COUNTYFP10"], r1))
        print(i["NAME10"])

    return all_maps

In [ ]:
x = create_all_county_maps(r_map[2], 49)

In [ ]:
import math

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    r = 6371  # Radius of earth in kilometers
    distance = r * c

    return distance


In [ ]:
def create_graph_from_2darray(arr):
    G = nx.Graph()
    for i in range(0, len(arr[0])):
        if arr[3][i] != 0:
          G.add_node(i, pos=(arr[0][i], arr[1][i]), prop=arr[2][i], pop=arr[3][i])
    for node,data in G.nodes(data=True):
        for node2,data2 in G.nodes(data=True):
            if haversine_distance(data['pos'][0], data['pos'][1], data2['pos'][0], data2['pos'][1]) <= 1:
                G.add_edge(node, node2)
    return G

In [ ]:
#For this matrix then you need to multiply by the proportion of all the connections.

def get_prob_matrix(adj_matrix):
    prob_matrix = np.zeros((adj_matrix.shape[0], adj_matrix.shape[0]))
    for i in range(adj_matrix.shape[0]):
        sum = 0
        for j in range(adj_matrix.shape[1]):
            sum += adj_matrix[i][j]
        for j in range(adj_matrix.shape[1]):
            prob_matrix[i][j] = (adj_matrix[i][j] / sum)
    return prob_matrix

def get_pop_density_matrix(adj_matrix, total_pops_array):
    prob_matrix = np.zeros((adj_matrix.shape[0], adj_matrix.shape[0]))
    for i in range(adj_matrix.shape[0]):
        sum = 0
        for j in range(adj_matrix.shape[1]):
            sum += (adj_matrix[i][j] * total_pops_array[j])
        for j in range(adj_matrix.shape[1]):
            prob_matrix[i][j] = ((adj_matrix[i][j]* total_pops_array[j]) / sum)
    return prob_matrix


def get_relevant_matrices(G):
    props = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
    A = get_prob_matrix(nx.adjacency_matrix(G).toarray())
    val = 0
    for node,data in G.nodes(data=True):
        props[val][val] = data['prop']
        val = val + 1
    return A, props

def get_relevant_density_matrices(G):
    total_pops_array = np.zeros(G.number_of_nodes())
    x = 0
    for node,data in G.nodes(data=True):
        total_pops_array[x] = data['pop']
        x = x + 1
    props = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
    A = get_pop_density_matrix(nx.adjacency_matrix(G).toarray(), total_pops_array)
    val = 0
    for node,data in G.nodes(data=True):
        props[val][val] = data['prop']
        val = val + 1
    return A, props

In [ ]:
#Individually feed unconnected subgraphs into this method
#FIX FOR SPARSE MATRICES
'''
prop matrix is a diagonal matrix with each proportion there

soc_matrix is the social matrix with the represeenting interactions between every two nodes
'''
def get_discrete_matrix(prop_matrix):
    discrete_matrix = np.zeros((prop_matrix.shape[0],prop_matrix.shape[0]))
    for i in range(prop_matrix.shape[0]):
        for j in range(prop_matrix.shape[1]):
            if(prop_matrix[i][j] < 0.5):
                discrete_matrix[i][j] = 0
            else:
                discrete_matrix[i][j] = 1
    return discrete_matrix


def Fryer_SSI(soc_matrix, prop_matrix):
    disc_prop_matrix = get_discrete_matrix(prop_matrix)
    alt_g = nx.Graph()
    nums = 0
    for i in range(len(disc_prop_matrix)):
        if disc_prop_matrix[i][i] == 1:
            alt_g.add_node(i)
            nums +=1
            for j in range(0,i):
                if disc_prop_matrix[j][j] == 1 and soc_matrix[i][j] > 0:
                    alt_g.add_edge(i,j)
    S = [alt_g.subgraph(c).copy() for c in nx.connected_components(alt_g)]
    total = 0
    for i in S:
      locs = []
      for a in i.nodes():
        locs.append(a)
      submatrix = soc_matrix[np.ix_(locs, locs)]
      total+= (np.real(max(np.linalg.eigvals(submatrix))) * len(locs))
    if nums == 0:
        return 0
    return (total / nums)

In [ ]:
import numpy as np

def PSSI(soc_matrix, prop_matrix, pop_matrix, return_vector=False):
    """
    Compute the population-weighted segregation index.

    soc_matrix: (n, n) array of nonnegative social weights f(i -> j).
    prop_matrix: length-n vector or diagonal matrix of race proportions.
    pop_matrix: length-n vector or diagonal matrix of populations.
    return_vector: also return the normalized right eigenvector if True.

    Nodes with no reachable population receive an all-zero row in A.
    """
    F = np.asarray(soc_matrix, dtype=float)
    if F.ndim != 2 or F.shape[0] != F.shape[1]:
        raise ValueError("soc_matrix must be square.")

    n = F.shape[0]

    def as_vector(values, name):
        values = np.asarray(values, dtype=float)
        if values.shape == (n, n):
            if not np.all(values == np.diag(np.diag(values))):
                raise ValueError(f"{name} must be diagonal.")
            values = np.diag(values)
        if values.shape != (n,):
            raise ValueError(f"{name} must be a length-n vector or diagonal matrix.")
        return values

    p = as_vector(prop_matrix, "prop_matrix")
    D = as_vector(pop_matrix, "pop_matrix")

    if not all(np.all(np.isfinite(x)) for x in (F, p, D)):
        raise ValueError("All inputs must contain finite values.")
    if np.any(F < 0) or np.any(D < 0):
        raise ValueError("Social weights and populations must be nonnegative.")
    if np.any((p < 0) | (p > 1)):
        raise ValueError("Race proportions must lie between 0 and 1.")

    # denominator[i] = sum_k f(i -> k) * D(k)
    denominator = F @ D

    # numerator[i, j] = f(i -> j) * D(j) * p(j)
    numerator = F * (D * p)[None, :]
    A = np.divide(
        numerator,
        denominator[:, None],
        out=np.zeros_like(F),
        where=denominator[:, None] > 0,
    )

    if not np.any(A):
        index = 0.0
        v = np.full(n, 1.0 / n) if n else np.array([])
    elif return_vector:
        eigenvalues, eigenvectors = np.linalg.eig(A)
        # A is nonnegative, so its spectral radius is a real eigenvalue.
        k = np.argmax(eigenvalues.real)
        index = float(eigenvalues[k].real)

        # Remove arbitrary eigenvector sign and numerical roundoff.
        v = np.abs(eigenvectors[:, k].real)
        v /= v.sum()
    else:
        index = float(np.linalg.eigvals(A).real.max())

    return (index, v) if return_vector else index

In [ ]:
import math
from datetime import datetime, timezone

import requests


def social_weight(
    origin,
    destination,
    *,
    api_key,
    beta,
    omega,
    c0,
    modes=("WALK", "TRANSIT"),
    driving_usd_per_km=None,
    parking_usd=0.0,
    bicycle_fee_usd=0.0,
    transit_fare_usd=None,
    driving_tolls_usd=None,
    departure_time=None,
):
    """
    Compute:
        C_ij = max(c0, min_m(price_m + omega * time_m))
        f_ij = exp(-beta * C_ij)

    Inputs:
        origin, destination: (latitude, longitude) in WGS84 degrees
        beta: cost sensitivity, in 1/USD
        omega: value of time, in USD/hour
        c0: local encounter cost floor, in USD
        modes: any combination of WALK, BICYCLE, DRIVE, TRANSIT
        departure_time: timezone-aware datetime; defaults to now

    Monetary assumptions:
        WALK: zero out-of-pocket cost
        BICYCLE: bicycle_fee_usd
        TRANSIT: API fare, unless transit_fare_usd overrides it
        DRIVE: distance * driving_usd_per_km + parking + tolls

    Missing fares and unknown toll prices raise errors.
    Walking/cycling routes containing ferries raise errors because their
    prices and scheduled waiting times cannot be inferred reliably.

    Minimizes over the routes returned by Google, not all possible routes.
    Each requested mode makes one API request.
    """

    def nonnegative(name, value):
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"{name} must be finite and nonnegative.")

    def waypoint(coords):
        if len(coords) != 2:
            raise ValueError("Use coordinates in (latitude, longitude) order.")
        lat, lon = map(float, coords)
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError("Invalid latitude or longitude.")
        return {
            "location": {
                "latLng": {"latitude": lat, "longitude": lon}
            }
        }

    def usd(money):
        if money.get("currencyCode") != "USD":
            raise ValueError("API price is not in USD; convert it explicitly.")
        amount = (
            float(money.get("units", 0))
            + float(money.get("nanos", 0)) / 1e9
        )
        nonnegative("API price", amount)
        return amount

    start = waypoint(origin)
    end = waypoint(destination)

    for name, value in {
        "beta": beta,
        "omega": omega,
        "c0": c0,
        "parking_usd": parking_usd,
        "bicycle_fee_usd": bicycle_fee_usd,
        "driving_usd_per_km": driving_usd_per_km,
        "transit_fare_usd": transit_fare_usd,
        "driving_tolls_usd": driving_tolls_usd,
    }.items():
        if value is not None:
            nonnegative(name, value)

    if beta == 0 or c0 == 0:
        raise ValueError("beta and c0 must be positive.")

    modes = tuple(dict.fromkeys(modes))
    allowed = {"WALK", "BICYCLE", "DRIVE", "TRANSIT"}
    if not modes or not set(modes).issubset(allowed):
        raise ValueError(f"Choose modes from {sorted(allowed)}.")

    departure = departure_time or datetime.now(timezone.utc)
    if departure.tzinfo is None or departure.utcoffset() is None:
        raise ValueError("departure_time must include a timezone.")
    timestamp = departure.astimezone(timezone.utc).isoformat().replace(
        "+00:00", "Z"
    )

    if start == end:
        return {
            "f": math.exp(-beta * c0),
            "log_f": -beta * c0,
            "C_usd": c0,
            "best_mode": "LOCAL",
            "routes": [],
            "unavailable_modes": [],
            "departure_time": timestamp,
        }

    if not api_key:
        raise ValueError("A Google Routes API key is required.")
    if "DRIVE" in modes and driving_usd_per_km is None:
        raise ValueError("Supply driving_usd_per_km to include DRIVE.")

    candidates = []
    unavailable = []

    for mode in modes:
        body = {
            "origin": start,
            "destination": end,
            "travelMode": mode,
            "computeAlternativeRoutes": True,
        }
        fields = ["routes.duration", "routes.distanceMeters"]

        if mode in {"WALK", "BICYCLE"}:
            fields.append(
                "routes.legs.steps.navigationInstruction.maneuver"
            )

        if mode in {"TRANSIT", "DRIVE"}:
            body["departureTime"] = timestamp

        if mode == "TRANSIT":
            fields.append("routes.travelAdvisory.transitFare")

        if mode == "DRIVE":
            body["routingPreference"] = "TRAFFIC_AWARE"
            body["extraComputations"] = ["TOLLS"]
            fields.append("routes.travelAdvisory.tollInfo")

        response = requests.post(
            "https://routes.googleapis.com/directions/v2:computeRoutes",
            headers={
                "Content-Type": "application/json",
                "X-Goog-Api-Key": api_key,
                "X-Goog-FieldMask": ",".join(fields),
            },
            json=body,
            timeout=30,
        )

        if not response.ok:
            raise RuntimeError(
                f"Google Routes returned HTTP {response.status_code}. "
                "Check API access, billing, quota, and request parameters."
            )

        routes = response.json().get("routes", [])
        if not routes:
            unavailable.append(mode)
            continue

        for route in routes:
            if mode in {"WALK", "BICYCLE"}:
                has_ferry = any(
                    step.get("navigationInstruction", {})
                    .get("maneuver", "").startswith("FERRY")
                    for leg in route.get("legs", [])
                    for step in leg.get("steps", [])
                )
                if has_ferry:
                    raise ValueError(
                        f"{mode} route includes a ferry. "
                        "Evaluate this pair with modes=('TRANSIT',) "
                        "to use scheduled transit times and fares."
                    )

            duration = route.get("duration", "")
            if not duration.endswith("s"):
                raise RuntimeError("API route is missing a valid duration.")

            hours = float(duration[:-1]) / 3600
            distance_m = float(route.get("distanceMeters", 0))
            nonnegative("route duration", hours)
            nonnegative("route distance", distance_m)
            advisory = route.get("travelAdvisory", {})

            if mode == "WALK":
                price = 0.0

            elif mode == "BICYCLE":
                price = bicycle_fee_usd

            elif mode == "TRANSIT":
                if transit_fare_usd is not None:
                    price = transit_fare_usd
                elif "transitFare" in advisory:
                    price = usd(advisory["transitFare"])
                else:
                    raise ValueError(
                        "Transit fare unavailable. Supply a verified "
                        "transit_fare_usd; missing fares are not free."
                    )

            else:  # DRIVE
                if driving_tolls_usd is not None:
                    tolls = driving_tolls_usd
                elif "tollInfo" not in advisory:
                    tolls = 0.0
                else:
                    estimates = advisory["tollInfo"].get(
                        "estimatedPrice", []
                    )
                    if not estimates:
                        raise ValueError(
                            "Toll price unavailable. Supply a verified "
                            "driving_tolls_usd."
                        )
                    tolls = sum(usd(item) for item in estimates)

                price = (
                    distance_m / 1000 * driving_usd_per_km
                    + parking_usd
                    + tolls
                )

            candidates.append({
                "mode": mode,
                "minutes": hours * 60,
                "distance_km": distance_m / 1000,
                "price_usd": price,
                "generalized_cost_usd": price + omega * hours,
            })

    if candidates:
        best = min(
            candidates, key=lambda route: route["generalized_cost_usd"]
        )
        C = max(c0, best["generalized_cost_usd"])
        best_mode = best["mode"]
    else:
        C = math.inf
        best_mode = None

    return {
        "f": math.exp(-beta * C),
        "log_f": -beta * C,
        "C_usd": C,
        "best_mode": best_mode,
        "routes": candidates,
        "unavailable_modes": unavailable,
        "departure_time": timestamp,
    }

In [ ]:
from getpass import getpass

api_key = getpass("Google Routes API key: ")

result = social_weight(
    origin=(40.8075, -73.9626),       # Columbia University
    destination=(40.7506, -73.9935),  # Penn Station
    api_key=api_key,
    beta=0.1,       # illustrative: 1/USD
    omega=15.0,     # illustrative: USD/hour
    c0=1.25,       # five minutes valued at $15/hour
    modes=("WALK", "TRANSIT"),
)

print(f"f(i → j): {result['f']:.6f}")
print(f"Generalized cost: ${result['C_usd']:.2f}")
print(f"Best mode: {result['best_mode']}")

In [ ]:
PSSI